# Part 4 — SARIMAX Model
**Appliance Energy Use Forecasting — 7PAM2033**

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/DolapoMichael/Time-Series-coding-Case-study-and-Report/blob/main/notebooks/04_sarimax.ipynb)

**⚠️ Runtime warning: the grid search in this notebook takes roughly 15–25 minutes to run end to end** (147 non-seasonal fits + 8 seasonal refinement fits + one larger final fit). This isn't a bug or a stuck cell — SARIMAX fitting is genuinely this slow on ~3,000 hourly observations. Progress prints every 25 combinations so you can see it moving.

**Why two stages, not one full seasonal grid:** the brief asks to loop over all combinations of p=[0,6], d=[0,2], q=[0,6] by AIC. A single SARIMAX fit *with* daily seasonality (period 24) takes ~30s on its own; looping all 147 combinations with seasonality included would take over an hour. Instead: Stage 1 grid-searches the exact p=[0,6]/d=[0,2]/q=[0,6] space on a **non-seasonal** ARIMA (fast — no seasonal term), then Stage 2 takes the best (p,d,q) and grid-searches a small seasonal space (P,D,Q ∈ {0,1}, period 24) on top of it — which also doubles as the project's explicit hyperparameter-tuning step.

In [ ]:
!pip install -q statsmodels

In [ ]:
import itertools
import time
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from statsmodels.graphics.tsaplots import plot_acf
from statsmodels.tsa.statespace.sarimax import SARIMAX

warnings.filterwarnings('ignore')  # expected convergence warnings from poor candidate orders during grid search

RAW_CSV_URL = 'https://raw.githubusercontent.com/LuisM78/Appliances-energy-prediction-data/master/energydata_complete.csv'
TARGET = 'Appliances'
DAILY_PERIOD = 24
TEST_DAYS = 14
SEASONAL_PERIOD = 24

DATA_DIR = Path('data')
OUTPUT_DIR = Path('outputs')
for d in [DATA_DIR, OUTPUT_DIR / 'forecasts', OUTPUT_DIR / 'metrics', OUTPUT_DIR / 'figures', OUTPUT_DIR / 'model_objects']:
    d.mkdir(parents=True, exist_ok=True)

plt.rcParams.update({'figure.dpi': 100, 'axes.grid': True, 'grid.alpha': 0.3})

## Load hourly data and split (self-contained)

In [ ]:
hourly_path = DATA_DIR / 'energydata_hourly.csv'

if hourly_path.exists():
    hourly = pd.read_csv(hourly_path, index_col=0, parse_dates=True)
else:
    print('No local hourly dataset found — rebuilding from the raw source...')
    raw = pd.read_csv(RAW_CSV_URL)
    raw['date'] = pd.to_datetime(raw['date'])
    raw = raw.set_index('date').sort_index()
    energy_cols = ['Appliances', 'lights']
    sensor_cols = [c for c in raw.columns if c not in energy_cols + ['rv1', 'rv2']]
    hourly = pd.concat([
        raw[energy_cols].resample('h').sum(),
        raw[sensor_cols].resample('h').mean(),
    ], axis=1)
    hourly.to_csv(hourly_path)

def train_test_split_by_days(series, test_days=TEST_DAYS):
    test_steps = test_days * DAILY_PERIOD
    return series.iloc[:-test_steps], series.iloc[-test_steps:]

def evaluate_forecast(name, y_true, y_pred, y_train, seasonality=DAILY_PERIOD):
    y_pred = y_pred.reindex(y_true.index)
    valid = y_true.notna() & y_pred.notna()
    yt, yp = y_true.loc[valid], y_pred.loc[valid]
    y_train_f = y_train.astype(float)
    naive_err = np.abs(y_train_f.iloc[seasonality:].values - y_train_f.iloc[:-seasonality].values)
    scale = naive_err.mean()
    return {
        'model': name,
        'MAE': float(np.mean(np.abs(yt.values - yp.values))),
        'RMSE': float(np.sqrt(np.mean((yt.values - yp.values) ** 2))),
        'MASE': float(np.mean(np.abs(yt.values - yp.values)) / scale) if scale else float('nan'),
        'Bias': float(np.mean(yp.values - yt.values)),
        'n_points': int(valid.sum()),
    }

series = hourly[TARGET].asfreq('h')
train, test = train_test_split_by_days(series, TEST_DAYS)
print(f'Train: {train.index.min()} to {train.index.max()} ({len(train)} obs)')
print(f'Test:  {test.index.min()} to {test.index.max()} ({len(test)} obs)')

## Stage 1 — non-seasonal grid search: p=[0,6], d=[0,2], q=[0,6] by AIC
(~7 minutes)

In [ ]:
def grid_search_nonseasonal(train, p_range=range(0, 7), d_range=range(0, 3), q_range=range(0, 7)):
    rows = []
    combos = list(itertools.product(p_range, d_range, q_range))
    t0 = time.time()
    for i, (p, d, q) in enumerate(combos):
        try:
            model = SARIMAX(train, order=(p, d, q), trend='c',
                             enforce_stationarity=False, enforce_invertibility=False)
            fit = model.fit(disp=False, maxiter=50)
            aic = fit.aic
        except Exception:
            aic = np.inf
        rows.append({'p': p, 'd': d, 'q': q, 'AIC': aic})
        if (i + 1) % 25 == 0:
            print(f'  ...{i + 1}/{len(combos)} combinations, {time.time() - t0:.0f}s elapsed')
    print(f'Done: {len(combos)} combinations in {time.time() - t0:.0f}s')
    return pd.DataFrame(rows).sort_values('AIC').reset_index(drop=True)

nonseasonal_results = grid_search_nonseasonal(train)
nonseasonal_results.to_csv(OUTPUT_DIR / 'metrics' / 'sarimax_nonseasonal_grid.csv', index=False)
best_pdq = nonseasonal_results.iloc[0]
order = (int(best_pdq.p), int(best_pdq.d), int(best_pdq.q))
print(f'Best non-seasonal order: {order}  (AIC={best_pdq.AIC:.1f})')

## Stage 2 — seasonal refinement: P,D,Q ∈ {0,1}, period 24
(~11 minutes — this is also the project's hyperparameter-tuning step)

In [ ]:
def grid_search_seasonal(train, order, period=SEASONAL_PERIOD):
    rows = []
    t0 = time.time()
    for P, D, Q in itertools.product([0, 1], [0, 1], [0, 1]):
        try:
            model = SARIMAX(train, order=order, seasonal_order=(P, D, Q, period), trend='c',
                             enforce_stationarity=False, enforce_invertibility=False)
            fit = model.fit(disp=False, maxiter=75)
            aic = fit.aic
        except Exception:
            aic = np.inf
        rows.append({'P': P, 'D': D, 'Q': Q, 'period': period, 'AIC': aic})
        print(f'  seasonal_order=({P},{D},{Q},{period}) -> AIC={aic:.1f}  ({time.time() - t0:.0f}s elapsed)')
    return pd.DataFrame(rows).sort_values('AIC').reset_index(drop=True)

seasonal_results = grid_search_seasonal(train, order)
seasonal_results.to_csv(OUTPUT_DIR / 'metrics' / 'sarimax_seasonal_grid.csv', index=False)
best_s = seasonal_results.iloc[0]
seasonal_order = (int(best_s.P), int(best_s.D), int(best_s.Q), SEASONAL_PERIOD)
print(f'Best seasonal order: {seasonal_order}  (AIC={best_s.AIC:.1f})')

## Fit the final model
(~3 minutes)

In [ ]:
t0 = time.time()
final_model = SARIMAX(train, order=order, seasonal_order=seasonal_order, trend='c',
                       enforce_stationarity=False, enforce_invertibility=False)
final_fit = final_model.fit(disp=False, maxiter=150)
print(f'Fit complete in {time.time() - t0:.0f}s. AIC={final_fit.aic:.1f}')
print(final_fit.summary().tables[0])

## Residual diagnostics

In [ ]:
resid = final_fit.resid.dropna()
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
plot_acf(resid, lags=48, ax=axes[0])
axes[0].set_title('ACF of residuals')
axes[1].hist(resid, bins=50, color='#1f5b8a', edgecolor='white')
axes[1].set_title(f'Residual distribution (mean={resid.mean():.1f}, std={resid.std():.1f})')
fig.tight_layout()
fig.savefig(OUTPUT_DIR / 'figures' / '04a_sarimax_residuals.png')
plt.show()

## Literal 24-hour-ahead forecast with 95% confidence interval

In [ ]:
fc_24h = final_fit.get_forecast(steps=DAILY_PERIOD)
mean_24h = fc_24h.predicted_mean
mean_24h.index = test.index[:DAILY_PERIOD]
conf_int_24h = fc_24h.conf_int(alpha=0.05)
conf_int_24h.index = mean_24h.index
actual_24h = test.iloc[:DAILY_PERIOD]
rmse_24h = float(np.sqrt(np.mean((actual_24h.values - mean_24h.values) ** 2)))
print(f'24h-ahead RMSE: {rmse_24h:.1f} Wh')

fig, ax = plt.subplots(figsize=(12, 4))
train.tail(3 * DAILY_PERIOD).plot(ax=ax, label='Train (last 3 days)', color='#999999')
actual_24h.plot(ax=ax, label='Actual (next 24h)', color='black', linewidth=1.8)
mean_24h.plot(ax=ax, label='SARIMAX forecast', color='#c0392b', linewidth=1.6)
ax.fill_between(conf_int_24h.index, conf_int_24h.iloc[:, 0], conf_int_24h.iloc[:, 1],
                color='#c0392b', alpha=0.15, label='95% CI')
ax.set_title('SARIMAX — literal 24-hour-ahead forecast with confidence interval')
ax.set_xlabel('Date'); ax.set_ylabel('Appliances (Wh / hour)')
ax.legend()
fig.tight_layout()
fig.savefig(OUTPUT_DIR / 'figures' / '04b_sarimax_24h_forecast.png')
plt.show()

## Full 336-hour continuous test-period forecast
(for comparability with Part 3's benchmarks)

In [ ]:
fc_full = final_fit.get_forecast(steps=len(test))
mean_full = fc_full.predicted_mean
mean_full.index = test.index
mean_full.name = 'sarimax'

metrics_full = evaluate_forecast('sarimax', test, mean_full, train)
print('SARIMAX metrics over the full 336h test period:')
pd.Series(metrics_full)

In [ ]:
fig, ax = plt.subplots(figsize=(14, 5))
train.tail(7 * DAILY_PERIOD).plot(ax=ax, label='Train (last 7 days)', color='#999999', linewidth=1)
test.plot(ax=ax, label='Actual (test)', color='black', linewidth=1.6)
mean_full.plot(ax=ax, label='SARIMAX (336h continuous)', color='#c0392b', linewidth=1.2, alpha=0.9)
ax.set_title('SARIMAX vs. actual — full 336h continuous test-period forecast')
ax.set_xlabel('Date'); ax.set_ylabel('Appliances (Wh / hour)')
ax.legend()
fig.tight_layout()
fig.savefig(OUTPUT_DIR / 'figures' / '04c_sarimax_full_test_forecast.png')
plt.show()

## Save outputs

In [ ]:
pd.DataFrame([metrics_full]).to_csv(OUTPUT_DIR / 'metrics' / 'sarimax_metrics.csv', index=False)
pd.DataFrame({'actual': test, 'sarimax': mean_full}).to_csv(OUTPUT_DIR / 'forecasts' / 'sarimax_forecasts.csv')
final_fit.save(str(OUTPUT_DIR / 'model_objects' / 'sarimax_final.pkl'))
print('Saved forecasts, metrics, and the fitted model to outputs/')